> **ARCHIV, NICHT MEHR AKTUELL:** Dies ist die frühere kombinierte Fassung von TP2, TP3 und TP4. Sie wird nur als Sicherung aufbewahrt und enthält noch nicht die Korrekturen der aktuellen Fassung. Bitte benutzen Sie stattdessen die drei Notebooks `TP2_Audioanalyse.ipynb`, `TP3_Denoising_Equalizer.ipynb` und `TP4_Spektrogramm_Sprachaktivitaet.ipynb` im Hauptordner.

# Audio-DSP-Labor: TP2, TP3 und TP4
### Digitale Audiosignalverarbeitung mit Python

Dieses Notebook führt Sie Schritt für Schritt in die digitale Audiosignalverarbeitung ein. Sie brauchen **keine Vorkenntnisse** in Python oder Maschinellem Lernen.

| Teil | Thema |
|---|---|
| Teil 0 | Erste Schritte mit Python |
| TP2 | Audiodateien verstehen und analysieren |
| TP3 | Rauschen reduzieren und Klang verbessern |
| TP4 | Sprache und Musik im Spektrogramm |

Wir arbeiten mit **drei echten Aufnahmen**: Sprache (`speech.wav`), Musik in Stereo (`music.wav`) und einem Umgebungsgeräusch, einem vorbeifahrenden Rennwagen (`sound.wav`). Quellen und Lizenzen stehen am Ende des Notebooks.

**So arbeiten Sie**
- Führen Sie die Zellen **der Reihe nach von oben nach unten** aus.
- Ändern Sie in den Aufgaben immer nur **ein oder zwei Parameter** und führen Sie die Zelle danach erneut aus.
- Hören Sie mit **Kopfhörern**. Tiefe Töne (zum Beispiel 50 Hz) sind über Laptop-Lautsprecher kaum hörbar.
- Schreiben Sie Ihre Beobachtungen direkt in die Antwort-Zellen (Doppelklick auf die Zelle zum Bearbeiten).

**Hinweise für Google Colab**
- Eventuell müssen Sie sich zuerst mit einem Google-Konto anmelden.
- Beim ersten Ausführen kann eine Warnung erscheinen, dass das Notebook nicht von Google stammt. Bestätigen Sie mit **"Trotzdem ausführen"** (englisch *Run anyway*).
- Nach einem Neustart oder Zurücksetzen der Laufzeit sind alle Variablen weg. In Colab sind dann auch die heruntergeladenen Dateien weg. Führen Sie einfach wieder alle Zellen von oben aus (Menü **Laufzeit**, Befehl zum Ausführen aller vorherigen Zellen, englisch *Runtime > Run before*). Die Audiodateien werden dabei automatisch neu geladen.
- Eine GPU wird nicht benötigt.

## Einrichtung (einmal am Anfang ausführen)

Die nächsten zwei Zellen bereiten alles vor. Sie müssen den Code hier noch nicht verstehen.

1. Die erste Zelle prüft, ob alle benötigten Python-Pakete vorhanden sind. Nur fehlende Pakete werden installiert.
2. Die zweite Zelle lädt die drei Audiodateien in den Ordner `data/`, **aber nur, wenn sie dort noch fehlen**. Liegt das Notebook zusammen mit dem Ordner `data/` auf Ihrem Rechner, werden die vorhandenen Dateien benutzt.

In [ ]:
# Einrichtung 1: fehlende Pakete installieren (in Google Colab ist meist schon alles da)
import importlib.util
import subprocess
import sys

pakete = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
          "soundfile": "soundfile", "librosa": "librosa", "IPython": "ipython"}

fehlend = [paket for modul, paket in pakete.items() if importlib.util.find_spec(modul) is None]
if fehlend:
    print("Installiere:", ", ".join(fehlend))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *fehlend])
    print("Fertig.")
else:
    print("Alle Pakete sind bereits installiert.")

In [ ]:
# Einrichtung 2: Audiodateien bereitstellen (Download nur, wenn eine Datei fehlt)
import os
import urllib.request
import soundfile as sf

DATEN_URL = "https://raw.githubusercontent.com/Lamhour-Mohamed-Akram/audio-dsp-lab/afe9e3d4fc8266c132421123ecd92235bc046779/data/"
ERWARTET = {"speech.wav": (16000, 1), "music.wav": (44100, 2), "sound.wav": (22050, 2)}   # (Abtastrate, Kanäle)


def audiodatei_ok(pfad, abtastrate, kanaele):
    """Prüft, ob die Datei eine lesbare WAV-Datei mit dem erwarteten Format ist."""
    try:
        info = sf.info(pfad)
    except Exception:
        return False
    return info.samplerate == abtastrate and info.channels == kanaele and info.frames > 0


os.makedirs("data", exist_ok=True)
for name, (abtastrate, kanaele) in ERWARTET.items():
    pfad = os.path.join("data", name)
    if audiodatei_ok(pfad, abtastrate, kanaele):
        print(name, ": vorhanden")
        continue
    print(name, ": wird heruntergeladen ...")
    try:
        with urllib.request.urlopen(DATEN_URL + name, timeout=60) as antwort:
            inhalt = antwort.read()
    except Exception as fehler:
        raise RuntimeError("Download von " + name + " fehlgeschlagen. Bitte Internetverbindung prüfen "
                           "und diese Zelle erneut ausführen. Grund: " + str(fehler))
    with open(pfad + ".tmp", "wb") as datei:
        datei.write(inhalt)
    if not audiodatei_ok(pfad + ".tmp", abtastrate, kanaele):
        os.remove(pfad + ".tmp")
        raise RuntimeError(name + " wurde geladen, ist aber keine gültige Audiodatei im erwarteten Format.")
    os.replace(pfad + ".tmp", pfad)
    print(name, ": OK")

print("Alle Audiodateien sind bereit.")

# Teil 0: Erste Schritte mit Python

Hier lernen Sie nur das, was Sie für das Labor brauchen.

## 0.1 Text-Zellen und Code-Zellen

Ein Notebook besteht aus **Zellen**:
- **Text-Zellen** (wie diese) enthalten Erklärungen.
- **Code-Zellen** enthalten Python-Code. Klicken Sie in die Zelle und drücken Sie **Shift + Enter**. Das Ergebnis erscheint direkt unter der Zelle.

Alles hinter `#` ist ein **Kommentar**. Python ignoriert Kommentare. Sie sind nur für Menschen gedacht.

In [ ]:
# Das ist ein Kommentar
print("Hallo! Meine erste Code-Zelle läuft.")

## 0.2 Variablen und Rechnen

Eine **Variable** ist ein Name für einen Wert. Mit `=` speichern Sie einen Wert unter diesem Namen. Python rechnet mit `+`, `-`, `*`, `/` und `**` (hoch).

**Die Reihenfolge zählt.** Eine Variable existiert erst, wenn die Zelle ausgeführt wurde, in der sie festgelegt wird. Wenn Sie eine Zelle überspringen, erscheint später ein Fehler wie `NameError`.

In [ ]:
abtastrate = 16000      # Werte pro Sekunde (Hz)
dauer = 2.5             # Sekunden (Kommazahlen schreibt man mit Punkt)

anzahl_samples = abtastrate * dauer
print("Anzahl der Werte:", anzahl_samples)

In [ ]:
# Diese Zelle benutzt die Variable "dauer" aus der Zelle darüber
print("Dauer in Millisekunden:", dauer * 1000)
print("2 hoch 10 =", 2 ** 10)

> **Aufgabe 0.1:** Ändern Sie `dauer` auf `10` und führen Sie **beide** Zellen erneut aus. Wie viele Werte hat eine 10 Sekunden lange Aufnahme mit 16000 Hz?

**Ihre Antwort:** 

## 0.3 Listen, Index ab 0 und Ausschnitte

Eine **Liste** speichert mehrere Werte in eckigen Klammern. Mit einem **Index** holen Sie einen einzelnen Wert heraus.

- Python beginnt beim Zählen mit **0**. Das erste Element ist `werte[0]`.
- `werte[-1]` ist das letzte Element.
- Ein **Ausschnitt** (englisch *slicing*) `werte[1:4]` enthält die Elemente mit Index 1, 2 und 3. Das Ende (4) gehört **nicht** dazu.

In [ ]:
werte = [0.0, 0.5, 1.0, 0.5, 0.0, -0.5, -1.0]

print("Ganze Liste:     ", werte)
print("Erstes Element:  ", werte[0])
print("Drittes Element: ", werte[2])
print("Letztes Element: ", werte[-1])
print("Ausschnitt [1:4]:", werte[1:4])
print("Anzahl Elemente: ", len(werte))

> **Aufgabe 0.2:** Ändern Sie die Zahlen in der eckigen Klammer so, dass genau die **ersten drei** Elemente ausgegeben werden.

In [ ]:
print(werte[0:1])    # ändern Sie die Zahlen

## 0.4 `import` und NumPy-Arrays

Python kann mit **Bibliotheken** erweitert werden. Mit `import` laden Sie eine Bibliothek. Für Zahlenreihen benutzen wir **NumPy** mit dem Kurznamen `np`.

Ein **NumPy-Array** ist wie eine Liste, aber man kann direkt damit rechnen: `x * 2` verdoppelt **jeden** Wert. Ein Audiosignal ist in Python genau so ein Array. Index und Ausschnitte funktionieren wie bei Listen.

In [ ]:
import numpy as np

x = np.array(werte)          # Liste in ein NumPy-Array umwandeln

print("x      =", x)
print("x * 2  =", x * 2)                       # jeder Wert mal 2
print("x[2:5] =", x[2:5])                      # Ausschnitt
print("Maximum:", np.max(x))
print("Größter Betrag:", np.max(np.abs(x)))    # abs = Betrag (ohne Vorzeichen)

## 0.5 `for`-Schleifen und `if`-Abfragen

- Eine **`for`-Schleife** wiederholt Code für jedes Element.
- Eine **`if`-Abfrage** führt Code nur aus, wenn eine Bedingung wahr ist. `else` gilt für alle anderen Fälle.
- Der Code **innerhalb** von `for` und `if` ist **eingerückt** (4 Leerzeichen). Daran erkennt Python, was dazugehört.

In [ ]:
schwelle = 0.4

for wert in werte:
    if wert > schwelle:
        print(wert, "ist laut")
    else:
        print(wert, "ist leise")

Oft zählt man mit `range(...)`. Mit `append` hängt man einen Wert hinten an eine Liste an.

In [ ]:
quadrate = []                  # eine leere Liste
for i in range(4):             # range(4) zählt 0, 1, 2, 3
    quadrate.append(i ** 2)    # Quadrat hinten anhängen
print(quadrate)

> **Aufgabe 0.3:** Setzen Sie oben `schwelle = 0.8` und führen Sie die Zelle erneut aus. Welche Werte gelten jetzt noch als "laut"? (Genau diese Idee benutzen wir in TP4, um Sprache zu erkennen.)

**Ihre Antwort:** 

## 0.6 Funktionen aufrufen

Eine **Funktion** ist ein fertiger Baustein. Man ruft sie mit ihrem Namen und runden Klammern auf: `name(eingabe)`. Mehrere Eingaben werden mit Komma getrennt. Manche Eingaben haben einen Namen, zum Beispiel `rate=16000`. Sie haben schon `print`, `len` und `np.max` benutzt.

In [ ]:
print(round(3.14159, 2))     # runden auf 2 Nachkommastellen
print(np.mean(x))            # Mittelwert des Arrays x
print(np.sqrt(16))           # Wurzel

## 0.7 Eigene Funktionen mit `def` und `return`

Mit `def` schreiben Sie eine eigene Funktion. `return` gibt das Ergebnis zurück. Auch hier ist der Inhalt eingerückt. Den Text in dreifachen Anführungszeichen nennt man **Beschreibung**. Er erklärt, was die Funktion tut.

In [ ]:
def sekunden_zu_samples(sekunden, abtastrate):
    """Rechnet eine Zeit in Sekunden in eine Anzahl von Samples um."""
    anzahl = int(sekunden * abtastrate)    # int() macht eine ganze Zahl daraus
    return anzahl

print(sekunden_zu_samples(2.0, 16000))
print(sekunden_zu_samples(0.5, 44100))

> **Aufgabe 0.4:** Wie viele Samples sind **25 Millisekunden** bei 16000 Hz? Ändern Sie die Zahl im Aufruf. (Tipp: 25 ms = 0.025 s)

In [ ]:
print(sekunden_zu_samples(0.0, 16000))    # ändern Sie 0.0

## 0.8 Kurz erklärt: Ein Ton ist eine Zahlenfolge

Ein Computer speichert Klang als **Folge von Zahlen** (Samples). Die **Abtastrate** sagt, wie viele Zahlen pro Sekunde gespeichert werden. Der einfachste Klang ist ein **Sinuston**. Wir erzeugen einen Ton mit 440 Hz und betrachten nur die ersten 10 Millisekunden.

Diagramme zeichnen wir mit der Bibliothek **matplotlib** (Kurzname `plt`).

*Synthetische Töne benutzen wir nur zur Erklärung. Die eigentlichen Übungen arbeiten mit echten Aufnahmen.*

In [ ]:
import matplotlib.pyplot as plt

sr_ton = 16000                                # Abtastrate in Hz
t = np.arange(sr_ton) / sr_ton                # Zeitachse: 1 Sekunde (0, 1/16000, 2/16000, ...)
ton = 0.3 * np.sin(2 * np.pi * 440 * t)       # Sinuston: 440 Hz, Amplitude 0.3

plt.figure(figsize=(10, 3))
plt.plot(t[:160] * 1000, ton[:160], marker=".")    # 160 Samples = 10 ms
plt.xlabel("Zeit [ms]")
plt.ylabel("Amplitude")
plt.title("Sinuston 440 Hz: jeder Punkt ist ein Sample")
plt.grid(True)
plt.show()

## 0.9 Eine echte Aufnahme plotten und anhören

Jetzt öffnen wir eine **echte Sprachaufnahme**.

- `sf.read(...)` aus der Bibliothek **soundfile** liest eine WAV-Datei. Die Funktion gibt **zwei** Dinge zurück: das Signal und die Abtastrate.
- `plt.plot(...)` zeichnet das Signal.
- Unsere kleine Funktion `anhoeren(...)` zeigt einen Abspieler. Sie verändert die Lautstärke **nicht** automatisch. So hören Sie echte Lautstärkeunterschiede zwischen verschiedenen Signalen. Mit `skala` kann man alle Signale gleich stark leiser machen (mehr dazu in TP2).

In [ ]:
from IPython.display import Audio, display

def anhoeren(signal, sr, text="", skala=1.0):
    """Zeigt einen Audio-Abspieler, ohne automatische Lautstärke-Anpassung."""
    signal = np.asarray(signal) * skala
    if np.max(np.abs(signal)) > 1:
        print("Achtung: Werte über 1 werden abgeschnitten (Übersteuerung). Bitte kleinere Skala wählen.")
        signal = np.clip(signal, -1, 1)
    if signal.ndim == 2:        # Stereo: der Abspieler erwartet (Kanäle, Samples)
        signal = signal.T
    if text:
        print(text)
    display(Audio(signal, rate=sr, normalize=False))

In [ ]:
sprache, sr_sprache = sf.read("data/speech.wav")

zeit = np.arange(len(sprache)) / sr_sprache     # Zeitachse in Sekunden

plt.figure(figsize=(10, 3))
plt.plot(zeit, sprache, linewidth=0.5)
plt.xlabel("Zeit [s]")
plt.ylabel("Amplitude")
plt.title("Sprachaufnahme speech.wav")
plt.grid(True)
plt.show()

anhoeren(sprache, sr_sprache, "Sprache (Original):")

> **Aufgabe 0.5:** Hören Sie die Aufnahme an. Wo im Diagramm sehen Sie **Pausen** zwischen den Sätzen? Nennen Sie zwei ungefähre Zeitpunkte.

**Ihre Antwort:** 

# TP2: Audiodateien verstehen und analysieren

## Lernziele

Nach diesem Versuch können Sie ...
- eine Audiodatei laden und **Abtastrate, Samples, Dauer, Amplitude und Kanäle** bestimmen,
- ein Signal **anhören** und als **Wellenform** darstellen, auch in einem kurzen Ausschnitt,
- **Stereo** und **Mono** unterscheiden,
- mit der **FFT** ein **Frequenzspektrum** berechnen und lesen,
- einen **Tiefpass** und einen **Hochpass** anwenden und die Wirkung hören und sehen.

## Vorbereitungsaufgaben

Bearbeiten Sie diese Fragen **vor** dem Labor. Kurze Antworten genügen.

1. Eine Aufnahme hat die Abtastrate 16000 Hz und dauert 3 Sekunden. Wie viele Samples hat sie?
2. Die höchste Frequenz, die ein digitales Signal enthalten kann, ist die halbe Abtastrate (**Nyquist-Frequenz**). Wie hoch ist sie bei 16000 Hz und bei 44100 Hz?
3. Was ist der Unterschied zwischen einer **Mono**-Aufnahme und einer **Stereo**-Aufnahme?
4. Was macht ein **Tiefpass**? Was macht ein **Hochpass**?

**Ihre Antworten:**
1.
2.
3.
4.

## Durchführung

### 2.1 Die drei Audiodateien laden

Jede WAV-Datei hat einen "Steckbrief":
- **Abtastrate** (Hz): Samples pro Sekunde
- **Kanäle**: 1 = Mono, 2 = Stereo
- **Samples**: Anzahl der gespeicherten Werte pro Kanal
- **Dauer** = Samples geteilt durch Abtastrate
- **Format**: `PCM_16` bedeutet 16-Bit-Ganzzahlen. Beim Laden rechnet `sf.read` sie in Kommazahlen zwischen -1 und +1 um.
- **Spitzenwert**: der größte Betrag im Signal. Werte bei 1.0 bedeuten **Übersteuerung** (englisch *clipping*).

In [ ]:
def steckbrief(dateiname):
    """Liest eine Audiodatei und zeigt ihre wichtigsten Eigenschaften."""
    info = sf.info("data/" + dateiname)
    signal, sr = sf.read("data/" + dateiname)
    print("Datei:       ", dateiname)
    print("Abtastrate:  ", sr, "Hz")
    print("Kanäle:      ", info.channels)
    print("Samples:     ", len(signal))
    print("Dauer:       ", round(len(signal) / sr, 2), "s")
    print("Format:      ", info.subtype)
    print("Spitzenwert: ", round(np.max(np.abs(signal)), 3))
    print("Übersteuert: ", np.sum(np.abs(signal) >= 0.999), "Samples")
    print()

for name in ["speech.wav", "music.wav", "sound.wav"]:
    steckbrief(name)

> **Aufgabe 2.1:** Übertragen Sie die Werte in **Tabelle A** der Auswertung. Welche Datei hat die höchste Abtastrate? Welche Dateien sind Stereo?

**Ihre Antwort:** 

Jetzt laden wir alle drei Aufnahmen in Variablen. `.shape` zeigt die **Form** eines Arrays:
- `(Samples,)` bei Mono: eine lange Zahlenreihe
- `(Samples, 2)` bei Stereo: eine Tabelle mit zwei Spalten, links und rechts

In [ ]:
sprache, sr_sprache = sf.read("data/speech.wav")
musik, sr_musik = sf.read("data/music.wav")
geraeusch, sr_geraeusch = sf.read("data/sound.wav")

print("Sprache: ", sprache.shape, "bei", sr_sprache, "Hz")
print("Musik:   ", musik.shape, "bei", sr_musik, "Hz")
print("Geräusch:", geraeusch.shape, "bei", sr_geraeusch, "Hz")

### 2.2 Einzelne Samples ansehen

Ein Signal ist nur eine lange Zahlenreihe. Um eine **Zeit** in einen **Index** umzurechnen, multipliziert man mit der Abtastrate: Index = Zeit in Sekunden mal Abtastrate. Dafür haben wir in Teil 0 die Funktion `sekunden_zu_samples` geschrieben.

In [ ]:
print("Die ersten 10 Samples:", np.round(sprache[:10], 4))

zeitpunkt_s = 1.5                                      # Zeitpunkt in Sekunden
index = sekunden_zu_samples(zeitpunkt_s, sr_sprache)
print("Index bei", zeitpunkt_s, "s:", index)
print("Wert an dieser Stelle:", round(sprache[index], 4))

> **Aufgabe 2.2:** Welcher Index gehört zu **1.5 s** in `music.wav` (44100 Hz)? Ersetzen Sie dazu in der Zelle `sr_sprache` durch `sr_musik` (nur in der Zeile mit `index = ...`). Warum ist der Index anders als bei der Sprache, obwohl die Zeit gleich ist?

**Ihre Antwort:** 

### 2.3 Anhören und Wellenformen vergleichen

Die **Wellenform** zeigt die Amplitude über der Zeit. Bei Stereo zeichnen wir beide Kanäle übereinander. `signal[:, 0]` bedeutet: alle Zeilen (`:`) aus Spalte 0, also der linke Kanal. `signal[:, 1]` ist der rechte Kanal.

In [ ]:
def wellenform(signal, sr, titel):
    """Zeichnet die Wellenform. Bei Stereo werden beide Kanäle gezeichnet."""
    zeit = np.arange(len(signal)) / sr
    plt.figure(figsize=(10, 2.5))
    if signal.ndim == 1:
        plt.plot(zeit, signal, linewidth=0.5)
    else:
        plt.plot(zeit, signal[:, 0], linewidth=0.5, label="links")
        plt.plot(zeit, signal[:, 1], linewidth=0.5, label="rechts", alpha=0.7)
        plt.legend(loc="upper right")
    plt.ylim(-1, 1)
    plt.xlabel("Zeit [s]")
    plt.ylabel("Amplitude")
    plt.title(titel)
    plt.grid(True)
    plt.show()

In [ ]:
wellenform(sprache, sr_sprache, "Sprache (mono)")
anhoeren(sprache, sr_sprache, "Sprache:")

wellenform(musik, sr_musik, "Musik (stereo)")
anhoeren(musik, sr_musik, "Musik:")

wellenform(geraeusch, sr_geraeusch, "Geräusch: Rennwagen (stereo)")
anhoeren(geraeusch, sr_geraeusch, "Geräusch:")

> **Aufgabe 2.3:** Beschreiben Sie für jede Aufnahme kurz die Wellenform (Pausen? gleichmäßig? lauter werdend?) und Ihren Höreindruck.

| Aufnahme | Wellenform | Höreindruck |
|---|---|---|
| Sprache | | |
| Musik | | |
| Geräusch | | |

### 2.4 In einen kurzen Abschnitt hineinzoomen

Mit einem Ausschnitt `signal[start:ende]` schneiden wir einen kurzen Teil heraus. Bei **50 ms** sieht man die einzelnen Schwingungen der Stimme.

In [ ]:
start_s = 4.0       # Beginn des Ausschnitts in Sekunden
laenge_s = 0.05     # Länge des Ausschnitts in Sekunden (0.05 s = 50 ms)

start = sekunden_zu_samples(start_s, sr_sprache)
ende = start + sekunden_zu_samples(laenge_s, sr_sprache)
ausschnitt = sprache[start:ende]

zeit_ms = np.arange(start, ende) / sr_sprache * 1000
plt.figure(figsize=(10, 3))
plt.plot(zeit_ms, ausschnitt)
plt.xlabel("Zeit [ms]")
plt.ylabel("Amplitude")
plt.title("Zoom in die Sprachaufnahme")
plt.grid(True)
plt.show()

> **Aufgabe 2.4:**
> a) Ändern Sie `start_s` so, dass der Ausschnitt in einer **Pause** liegt (siehe Aufgabe 0.5). Wie sieht die Wellenform jetzt aus? Achten Sie auf die Werte der y-Achse.
> b) Setzen Sie `start_s = 4.0` und `laenge_s = 0.5`. Warum erkennt man die einzelnen Schwingungen nun schlechter?

**Ihre Antwort:** 

Einen längeren Abschnitt können Sie auch anhören:

In [ ]:
von_s = 3.0     # Beginn in Sekunden
bis_s = 6.0     # Ende in Sekunden

stueck = sprache[sekunden_zu_samples(von_s, sr_sprache):sekunden_zu_samples(bis_s, sr_sprache)]
print("Länge des Stücks:", len(stueck) / sr_sprache, "s")
anhoeren(stueck, sr_sprache, "Ausgewählter Abschnitt:")

> **Aufgabe 2.5:** Wählen Sie `von_s` und `bis_s` so, dass genau **ein** Satzteil ohne Pause zu hören ist. Notieren Sie die Zeiten.

**Ihre Antwort:** 

### 2.5 Stereo und Mono

Eine **Stereo**-Aufnahme hat zwei Kanäle. Beim Hören mit Kopfhörern kommt der linke Kanal links und der rechte Kanal rechts an. Dadurch hören wir, wo ein Instrument steht.

- **Mono** entsteht, wenn man beide Kanäle mittelt: (links + rechts) / 2. Dann kommt aus beiden Seiten dasselbe Signal.
- Die **Differenz** (links - rechts) / 2 enthält nur das, was sich zwischen den Kanälen unterscheidet. Klänge, die genau in der Mitte stehen, verschwinden darin.
- Der **RMS-Wert** (Effektivwert) ist die Wurzel aus dem Mittelwert der quadrierten Samples. Er ist ein Maß für die mittlere Lautstärke.
- Die **Korrelation** misst, wie ähnlich zwei Kanäle sind: 1 = identisch, 0 = völlig verschieden.

Wir benutzen die ersten 8 Sekunden der Musik, damit das Notebook schnell bleibt.

In [ ]:
musik_kurz = musik[:sekunden_zu_samples(8, sr_musik)]     # erste 8 Sekunden, beide Kanäle

links = musik_kurz[:, 0]
rechts = musik_kurz[:, 1]
mono = (links + rechts) / 2
differenz = (links - rechts) / 2

print("RMS links: ", round(np.sqrt(np.mean(links**2)), 4))
print("RMS rechts:", round(np.sqrt(np.mean(rechts**2)), 4))
print("Korrelation links/rechts:", round(np.corrcoef(links, rechts)[0, 1], 3))

In [ ]:
nur_links = musik_kurz.copy()
nur_links[:, 1] = 0          # rechten Kanal stumm schalten

nur_rechts = musik_kurz.copy()
nur_rechts[:, 0] = 0         # linken Kanal stumm schalten

anhoeren(musik_kurz, sr_musik, "Original (stereo):")
anhoeren(nur_links, sr_musik, "Nur linker Kanal:")
anhoeren(nur_rechts, sr_musik, "Nur rechter Kanal:")
anhoeren(mono, sr_musik, "Mono-Mischung (links + rechts) / 2:")
anhoeren(differenz, sr_musik, "Differenz (links - rechts) / 2:")

In [ ]:
# Zum Vergleich: Wie ähnlich sind die beiden Kanäle beim Rennwagen?
print("Korrelation links/rechts beim Rennwagen:", round(np.corrcoef(geraeusch[:, 0], geraeusch[:, 1])[0, 1], 3))

> **Aufgabe 2.6:** Hören Sie mit Kopfhörern.
> a) Welche Instrumente hören Sie eher links, welche eher rechts?
> b) Was bleibt in der Differenz übrig? Was fehlt?
> c) Bei welcher Aufnahme sind linker und rechter Kanal ähnlicher, Musik oder Rennwagen? Haben Sie eine Erklärung?

**Ihre Antworten:**
a)
b)
c)

### 2.6 Das Frequenzspektrum mit der FFT

Die Wellenform zeigt das Signal über der **Zeit**. Das **Spektrum** zeigt, **welche Frequenzen** im Signal enthalten sind. Die **FFT** (Fast Fourier Transform) ist ein schnelles Rechenverfahren dafür.

- x-Achse: Frequenz in Hz
- y-Achse: Stärke in **dB** (Dezibel). Ein Sinus mit Amplitude 1 hat 0 dB. **-20 dB** bedeutet 10-mal kleinere Amplitude, **-40 dB** bedeutet 100-mal kleiner.

Die Funktion `spektrum` erledigt die Rechnung. Stereo wird dafür zu Mono gemittelt. Das Ergebnis wird leicht geglättet (über etwa 20 Hz), damit das Bild bei echten Aufnahmen lesbar bleibt.

In [ ]:
def spektrum(signal, sr, glaettung_hz=20):
    """Berechnet das Betragsspektrum in dB. Gibt (Frequenzen, Stärke in dB) zurück."""
    if signal.ndim == 2:
        signal = signal.mean(axis=1)                        # Stereo zu Mono (nur für die Analyse)
    betrag = np.abs(np.fft.rfft(signal)) * 2 / len(signal)  # FFT, skaliert auf die Amplitude
    frequenzen = np.fft.rfftfreq(len(signal), d=1 / sr)
    breite = int(glaettung_hz * len(signal) / sr)           # Glättung: Anzahl Frequenzpunkte
    if breite > 1:
        betrag = np.convolve(betrag, np.ones(breite) / breite, mode="same")
    return frequenzen, 20 * np.log10(betrag + 1e-12)

**Kurz erklärt mit einem Testton:** Zwei Sinustöne (440 Hz mit Amplitude 0.3 und 1000 Hz mit Amplitude 0.1) ergeben im Spektrum genau **zwei Spitzen**. Hier schalten wir die Glättung aus.

In [ ]:
testton = 0.3 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.sin(2 * np.pi * 1000 * t)   # t und sr_ton aus Teil 0.8
f_test, db_test = spektrum(testton, sr_ton, glaettung_hz=0)

plt.figure(figsize=(10, 3))
plt.plot(f_test, db_test)
plt.xlim(0, 2000)
plt.ylim(-80, 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Spektrum des Testtons: zwei Spitzen")
plt.grid(True)
plt.show()

**Jetzt echte Aufnahmen.** Echte Sprache und Musik bestehen **nicht** aus wenigen festen Tönen. Ihre Frequenzen **ändern sich ständig**: andere Laute, andere Noten, andere Instrumente. Die FFT über die **ganze Datei** mischt alle Momente zusammen. Deshalb sehen wir breite "Hügel" statt einzelner scharfer Spitzen.

Damit man die Form gut vergleichen kann, verschieben wir jede Kurve so, dass ihr Maximum bei 0 dB liegt.

In [ ]:
max_frequenz = 8000     # bis zu welcher Frequenz gezeigt wird (Hz)

f1, db1 = spektrum(sprache, sr_sprache)
f2, db2 = spektrum(musik, sr_musik)
f3, db3 = spektrum(geraeusch, sr_geraeusch)

plt.figure(figsize=(10, 4))
plt.plot(f1, db1 - np.max(db1), label="Sprache", linewidth=0.8)
plt.plot(f2, db2 - np.max(db2), label="Musik", linewidth=0.8)
plt.plot(f3, db3 - np.max(db3), label="Rennwagen", linewidth=0.8)
plt.xlim(0, max_frequenz)
plt.ylim(-80, 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB] (Maximum = 0 dB)")
plt.title("Spektrum der ganzen Dateien")
plt.legend()
plt.grid(True)
plt.show()

> **Aufgabe 2.7:**
> a) In welchem Frequenzbereich hat die **Sprache** die meiste Energie?
> b) Welche Aufnahme hat oberhalb von 4000 Hz am meisten Energie?
> c) Setzen Sie `max_frequenz = 20000`. Warum endet die Sprache bei 8000 Hz und der Rennwagen bei 11025 Hz? (Tipp: Vorbereitungsaufgabe 2)

**Ihre Antwort:** 

Nimmt man nur einen **kurzen** Ausschnitt eines Vokals (zum Beispiel 100 ms), dann ändert sich der Klang in dieser kurzen Zeit kaum. Jetzt sieht man auch bei Sprache wieder Spitzen: die **Grundfrequenz** der Stimme und ihre Vielfachen, die **Obertöne**.

In [ ]:
start_s = 4.0       # Beginn (hier liegt ein Vokal)
laenge_s = 0.1      # Länge in Sekunden

start = sekunden_zu_samples(start_s, sr_sprache)
ende = start + sekunden_zu_samples(laenge_s, sr_sprache)
f_kurz, db_kurz = spektrum(sprache[start:ende], sr_sprache)

plt.figure(figsize=(10, 3))
plt.plot(f_kurz, db_kurz)
plt.xlim(0, 2000)
plt.ylim(np.max(db_kurz) - 60, np.max(db_kurz) + 5)     # zeigt die oberen 60 dB
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Spektrum eines kurzen Ausschnitts (100 ms)")
plt.grid(True)
plt.show()

> **Aufgabe 2.8:** Schätzen Sie aus dem Diagramm den Abstand zwischen zwei benachbarten Spitzen. Das ist ungefähr die Grundfrequenz der Stimme. Ändern Sie danach `start_s` auf `5.0` und auf `2.5` (Pause). Was ändert sich?

**Ihre Antwort:** 

### 2.7 Tiefpass und Hochpass

Ein **Filter** verändert, welche Frequenzen durchkommen.

| Filter | lässt durch | dämpft | typische Wirkung |
|---|---|---|---|
| Tiefpass | tiefe Frequenzen | hohe Frequenzen | Klang wird dumpf |
| Hochpass | hohe Frequenzen | tiefe Frequenzen | Klang wird dünn, Brummen verschwindet |

- Die **Grenzfrequenz** legt fest, wo der Filter anfängt zu dämpfen.
- Wir benutzen einen fertigen Filterentwurf (**Butterworth**-Filter) aus der Bibliothek **scipy**.
- Die **Ordnung** bestimmt, wie steil der Filter ist.
- `sosfiltfilt` filtert vorwärts und rückwärts. Dadurch wird das Signal zeitlich nicht verschoben.
- `axis=0` sorgt dafür, dass bei Stereo jeder Kanal einzeln gefiltert wird. Das Stereobild bleibt erhalten.

In [ ]:
from scipy import signal as sig

def tiefpass(x, sr, grenzfrequenz, ordnung=4):
    """Butterworth-Tiefpass: dämpft Frequenzen oberhalb der Grenzfrequenz."""
    sos = sig.butter(ordnung, grenzfrequenz, btype="lowpass", fs=sr, output="sos")
    return sig.sosfiltfilt(sos, x, axis=0)

def hochpass(x, sr, grenzfrequenz, ordnung=4):
    """Butterworth-Hochpass: dämpft Frequenzen unterhalb der Grenzfrequenz."""
    sos = sig.butter(ordnung, grenzfrequenz, btype="highpass", fs=sr, output="sos")
    return sig.sosfiltfilt(sos, x, axis=0)

**Wie wirkt der Filter?** Wir schicken einen einzelnen kurzen Klick (einen **Impuls**) durch den Filter und berechnen das Spektrum der Antwort. Das Ergebnis heißt **Frequenzgang**: 0 dB bedeutet "geht ungehindert durch", -40 dB bedeutet "stark gedämpft". Bei der Grenzfrequenz ist das Signal hier um etwa 6 dB gedämpft.

Die x-Achse ist **logarithmisch**: 100 Hz, 1000 Hz und 10000 Hz haben gleichen Abstand.

In [ ]:
grenzfrequenz = 1000     # Hz

impuls = np.zeros(8192)
impuls[4096] = 1.0                                      # ein einzelner Klick
antwort_tp = tiefpass(impuls, sr_musik, grenzfrequenz)

f_gang, db_gang = spektrum(antwort_tp, sr_musik, glaettung_hz=0)
db_gang = db_gang - 20 * np.log10(2 / len(antwort_tp))  # auf 0 dB = ungedämpft umrechnen

plt.figure(figsize=(10, 3))
plt.semilogx(f_gang[1:], db_gang[1:])
plt.axvline(grenzfrequenz, color="red", linestyle="--", label="Grenzfrequenz")
plt.ylim(-80, 5)
plt.xlabel("Frequenz [Hz] (logarithmische Achse)")
plt.ylabel("Verstärkung [dB]")
plt.title("Frequenzgang des Tiefpasses")
plt.legend()
plt.grid(True)
plt.show()

**Faire Vergleiche beim Hören:** Wir spielen Original und gefiltertes Signal mit **demselben Faktor** ab. Nur wenn ein Signal sonst über 1 hinausgehen würde, machen wir **alle** Signale mit derselben `skala` leiser. So bleiben echte Lautstärkeunterschiede hörbar, und nichts übersteuert.

In [ ]:
def gemeinsame_skala(*signale):
    """Ein gemeinsamer Faktor für alle Signale. Er ist nur kleiner als 1, wenn sonst etwas übersteuern würde."""
    spitze = max(np.max(np.abs(s)) for s in signale)
    return min(1.0, 0.99 / spitze)

**Tiefpass auf Musik**

In [ ]:
grenzfrequenz = 1000      # Hz, ändern Sie diesen Wert

musik_tp = tiefpass(musik_kurz, sr_musik, grenzfrequenz)

skala = gemeinsame_skala(musik_kurz, musik_tp)
print("Gemeinsame Skala:", round(skala, 3))
anhoeren(musik_kurz, sr_musik, "Original:", skala)
anhoeren(musik_tp, sr_musik, "Tiefpass:", skala)

In [ ]:
f_orig, db_orig = spektrum(musik_kurz, sr_musik)
f_tp, db_tp = spektrum(musik_tp, sr_musik)

plt.figure(figsize=(10, 3))
plt.plot(f_orig, db_orig, label="Original", linewidth=0.8)
plt.plot(f_tp, db_tp, label="Tiefpass", linewidth=0.8)
plt.axvline(grenzfrequenz, color="red", linestyle="--", label="Grenzfrequenz")
plt.xlim(0, 10000)
plt.ylim(np.max(db_orig) - 80, np.max(db_orig) + 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Musik: Spektrum vor und nach dem Tiefpass")
plt.legend()
plt.grid(True)
plt.show()

> **Aufgabe 2.9:** Probieren Sie `grenzfrequenz = 300`, `1000` und `4000`. Führen Sie beide Zellen jeweils erneut aus. Tragen Sie Ihren Höreindruck in **Tabelle B** ein.

**Hochpass auf Sprache**

In [ ]:
grenzfrequenz_hp = 500     # Hz, ändern Sie diesen Wert

sprache_hp = hochpass(sprache, sr_sprache, grenzfrequenz_hp)

skala = gemeinsame_skala(sprache, sprache_hp)
anhoeren(sprache, sr_sprache, "Original:", skala)
anhoeren(sprache_hp, sr_sprache, "Hochpass:", skala)

f_orig, db_orig = spektrum(sprache, sr_sprache)
f_hp, db_hp = spektrum(sprache_hp, sr_sprache)
plt.figure(figsize=(10, 3))
plt.plot(f_orig, db_orig, label="Original", linewidth=0.8)
plt.plot(f_hp, db_hp, label="Hochpass", linewidth=0.8)
plt.axvline(grenzfrequenz_hp, color="red", linestyle="--", label="Grenzfrequenz")
plt.xlim(0, 4000)
plt.ylim(np.max(db_orig) - 80, np.max(db_orig) + 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Sprache: Spektrum vor und nach dem Hochpass")
plt.legend()
plt.grid(True)
plt.show()

> **Aufgabe 2.10:** Probieren Sie `grenzfrequenz_hp = 100`, `500` und `2000`. Ist die Sprache noch verständlich? Warum wird sie leiser? Tragen Sie Ihre Beobachtungen in **Tabelle B** ein.

## Auswertung TP2

**Tabelle A: Eigenschaften der Aufnahmen**

| Datei | Abtastrate [Hz] | Kanäle | Samples | Dauer [s] | Spitzenwert | Übersteuert? |
|---|---|---|---|---|---|---|
| speech.wav | | | | | | |
| music.wav | | | | | | |
| sound.wav | | | | | | |

**Tabelle B: Filter**

| Signal | Filter | Grenzfrequenz [Hz] | Höreindruck (dumpf, dünn, verständlich, ...) |
|---|---|---|---|
| Musik | Tiefpass | 300 | |
| Musik | Tiefpass | 1000 | |
| Musik | Tiefpass | 4000 | |
| Sprache | Hochpass | 100 | |
| Sprache | Hochpass | 500 | |
| Sprache | Hochpass | 2000 | |

**Fragen**
1. Warum reicht für Sprache eine Abtastrate von 16000 Hz, während Musik oft mit 44100 Hz gespeichert wird?
2. Warum zeigt das Spektrum der ganzen Sprachaufnahme keine einzelnen scharfen Spitzen wie der Testton?
3. Was hören Sie in der Differenz (links - rechts) der Musik? Warum?
4. Ab welcher Tiefpass-Grenzfrequenz klingt die Musik für Sie "normal"? Begründen Sie mit dem Spektrum.
5. Ein Hochpass mit 2000 Hz macht Sprache leiser und dünner. Welche Information geht dabei verloren?

**Ihre Antworten:**
1. 
2. 
3. 
4. 
5. 

# TP3: Rauschen reduzieren und Klang verbessern

## Lernziele

Nach diesem Versuch können Sie ...
- Störungen **reproduzierbar** zu einem sauberen Sprachsignal hinzufügen,
- ein **Brummen** mit einem passenden Filter entfernen,
- **breitbandiges Rauschen** mit der **spektralen Subtraktion** verringern,
- das **SNR** (Signal-Rausch-Verhältnis) korrekt berechnen und vergleichen,
- erklären, warum ein besseres SNR nicht immer **besser klingt**,
- Musik mit einem **Equalizer** verändern,
- den Unterschied zwischen **lauter** und **besser** erklären.

## Vorbereitungsaufgaben

1. Das **SNR** in dB ist: SNR = 10 mal log10(Leistung des Signals / Leistung des Rauschens). log10 ist der Logarithmus zur Basis 10. Wie groß ist das SNR, wenn Signal und Rauschen **gleich viel** Leistung haben? Und wenn das Signal **10-mal** so viel Leistung hat?
2. Das Stromnetz in Europa hat 50 Hz. Welche Frequenzen hat ein typisches **Netzbrummen** (Grundton und die ersten zwei Vielfachen)?
3. **Weißes Rauschen** enthält alle Frequenzen gleich stark. Warum kann ein Tiefpass es **nicht vollständig** entfernen?
4. Eine verrauschte Aufnahme wird **lauter** gemacht. Wird das Rauschen dadurch weniger störend?

**Ihre Antworten:**
1.
2.
3.
4.

## Durchführung

### 3.1 Saubere Referenz und SNR

`speech.wav` ist unsere **saubere Referenz**. Wir fügen selbst Störungen hinzu. Weil wir das saubere Signal kennen, können wir den Fehler genau messen:

- Fehler = bearbeitetes Signal - sauberes Signal
- **SNR in dB** = 10 mal log10(Leistung des sauberen Signals / Leistung des Fehlers)

Ein großes SNR bedeutet wenig Störung. **Jede** Abweichung vom sauberen Signal zählt als Fehler, auch wenn ein Filter nützliche Teile der Sprache entfernt.

*In der Praxis kennt man das saubere Signal meistens nicht. Dieser Versuch ist deshalb eine Laborsituation.*

In [ ]:
def snr_db(sauber, test):
    """SNR in dB im Vergleich zur sauberen Referenz."""
    fehler = test - sauber
    return 10 * np.log10(np.sum(sauber**2) / np.sum(fehler**2))

zeit_sprache = np.arange(len(sprache)) / sr_sprache    # Zeitachse in Sekunden

### 3.2 Störung 1: Netzbrummen

Wir addieren ein Brummen aus 50 Hz, 100 Hz und 150 Hz. Diese Störung ist **klar begrenzt**: Sie liegt nur bei tiefen Frequenzen.

In [ ]:
brummen = (0.03 * np.sin(2 * np.pi * 50 * zeit_sprache)
           + 0.02 * np.sin(2 * np.pi * 100 * zeit_sprache)
           + 0.01 * np.sin(2 * np.pi * 150 * zeit_sprache))

mit_brummen = sprache + brummen
print("SNR mit Brummen:", round(snr_db(sprache, mit_brummen), 1), "dB")

skala = gemeinsame_skala(sprache, mit_brummen)
anhoeren(sprache, sr_sprache, "Sauber:", skala)
anhoeren(mit_brummen, sr_sprache, "Mit Brummen:", skala)

In [ ]:
f_sauber, db_sauber = spektrum(sprache, sr_sprache, glaettung_hz=2)
f_brumm, db_brumm = spektrum(mit_brummen, sr_sprache, glaettung_hz=2)

plt.figure(figsize=(10, 3))
plt.plot(f_brumm, db_brumm, label="mit Brummen", linewidth=0.8)
plt.plot(f_sauber, db_sauber, label="sauber", linewidth=0.8)
plt.xlim(0, 1000)
plt.ylim(-110, -20)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Das Brummen erscheint als drei Spitzen bei 50, 100 und 150 Hz")
plt.legend()
plt.grid(True)
plt.show()

Das Brummen liegt unter 200 Hz. Ein **Hochpass** kann es deshalb entfernen. **Aber Achtung:** Die Stimme hat ebenfalls Anteile bei tiefen Frequenzen. Ist die Grenzfrequenz zu hoch, entfernt der Filter auch **nützliche Sprachinformation**. Wir benutzen hier einen steileren Filter (Ordnung 6).

In [ ]:
grenzfrequenz_brumm = 200    # Hz, ändern Sie diesen Wert

ohne_brummen = hochpass(mit_brummen, sr_sprache, grenzfrequenz_brumm, ordnung=6)

print("SNR vorher: ", round(snr_db(sprache, mit_brummen), 1), "dB")
print("SNR nachher:", round(snr_db(sprache, ohne_brummen), 1), "dB")

skala = gemeinsame_skala(sprache, mit_brummen, ohne_brummen)
anhoeren(sprache, sr_sprache, "Sauber:", skala)
anhoeren(mit_brummen, sr_sprache, "Mit Brummen:", skala)
anhoeren(ohne_brummen, sr_sprache, "Nach dem Hochpass:", skala)

> **Aufgabe 3.1:** Probieren Sie `grenzfrequenz_brumm = 60`, `120`, `200`, `400` und `800`. Tragen Sie SNR und Höreindruck in **Tabelle C** ein. Bei welcher Grenzfrequenz ist das SNR am größten? Warum sinkt es bei 800 Hz wieder?

**Zusatz (optional): Kerbfilter.** Ein **Kerbfilter** (englisch *notch*) entfernt nur einen sehr schmalen Frequenzbereich. Mit drei Kerbfiltern bei 50, 100 und 150 Hz bleibt der Rest der Stimme fast unverändert.

In [ ]:
kerb = mit_brummen.copy()
for frequenz in [50, 100, 150]:
    b, a = sig.iirnotch(frequenz, Q=30, fs=sr_sprache)     # Q groß = sehr schmale Kerbe
    kerb = sig.filtfilt(b, a, kerb)

print("SNR nach den Kerbfiltern:", round(snr_db(sprache, kerb), 1), "dB")
anhoeren(kerb, sr_sprache, "Nach den Kerbfiltern:", skala)

### 3.3 Störung 2: breitbandiges Rauschen

**Weißes Rauschen** (wie Rauschen im Radio zwischen zwei Sendern) enthält **alle Frequenzen**, also auch die Frequenzen der Sprache. Wir erzeugen es mit Zufallszahlen.

- Mit `seed=42` erzeugt der Zufallsgenerator **jedes Mal dasselbe** Rauschen. So sind Ihre Ergebnisse reproduzierbar und mit anderen Gruppen vergleichbar.
- `ziel_snr_db` legt fest, wie stark das Rauschen ist.
- Zusätzlich erzeugen wir eine **Rauschprobe** von 1 Sekunde. Das ist so, als hätte das Mikrofon vor dem Sprechen kurz nur das Rauschen aufgenommen. Diese Probe brauchen wir in 3.4.

In [ ]:
ziel_snr_db = 10                             # gewünschtes SNR in dB

rng = np.random.default_rng(seed=42)         # fester Startwert: reproduzierbares Rauschen
leistung_sprache = np.mean(sprache**2)
rausch_staerke = np.sqrt(leistung_sprache / 10**(ziel_snr_db / 10))

rauschen = rausch_staerke * rng.normal(0, 1, len(sprache))
rauschprobe = rausch_staerke * rng.normal(0, 1, sr_sprache)     # 1 Sekunde nur Rauschen

verrauscht = sprache + rauschen
print("SNR der verrauschten Sprache:", round(snr_db(sprache, verrauscht), 1), "dB")

skala = gemeinsame_skala(sprache, verrauscht)
anhoeren(sprache, sr_sprache, "Sauber:", skala)
anhoeren(verrauscht, sr_sprache, "Verrauscht:", skala)

**Hilft ein Tiefpass?** Ein Tiefpass entfernt nur das Rauschen **oberhalb** der Grenzfrequenz. Das Rauschen im Frequenzbereich der Sprache bleibt. Gleichzeitig verliert die Sprache hohe Anteile, zum Beispiel bei "s", "f" und "sch".

In [ ]:
grenzfrequenz_tp = 3000   # Hz, ändern Sie diesen Wert

verrauscht_tp = tiefpass(verrauscht, sr_sprache, grenzfrequenz_tp)

print("SNR verrauscht:       ", round(snr_db(sprache, verrauscht), 1), "dB")
print("SNR nach dem Tiefpass:", round(snr_db(sprache, verrauscht_tp), 1), "dB")
anhoeren(verrauscht, sr_sprache, "Verrauscht:", skala)
anhoeren(verrauscht_tp, sr_sprache, "Verrauscht und Tiefpass:", skala)

> **Aufgabe 3.2:** Probieren Sie `grenzfrequenz_tp = 1000`, `3000` und `6000`. Ist das Rauschen danach verschwunden? Was passiert mit der Sprache?

**Ihre Antwort:** 

### 3.4 Spektrale Subtraktion

Die Idee in vier Schritten:
1. Das verrauschte Signal wird in viele kurze Stücke (32 ms) zerlegt. Für jedes Stück wird ein Spektrum berechnet.
2. Aus der **Rauschprobe** wird berechnet, wie stark das Rauschen im Mittel bei jeder Frequenz ist.
3. Von jedem Spektrum wird diese Rauschstärke **abgezogen**. Ein kleiner **Boden** verhindert, dass Werte negativ werden.
4. Aus den bereinigten Spektren wird wieder ein Zeitsignal.

Das Verfahren funktioniert gut bei **gleichmäßigem** Rauschen, dessen Klang sich nicht ändert (stationäres Rauschen). Es entfernt **nicht** jede Art von Störung. Zu starkes Abziehen erzeugt Artefakte: Das sogenannte **musikalische Rauschen** klingt wie Zwitschern oder Blubbern.

Parameter:
- `staerke`: wie viel vom Rauschspektrum abgezogen wird (1.0 = genau die mittlere Rauschstärke)
- `boden`: welcher Anteil vom Spektrum mindestens übrig bleibt (0.05 = 5 %)

In [ ]:
import librosa

def spektrale_subtraktion(verrauscht, rauschprobe, staerke=1.0, boden=0.05):
    """Einfache spektrale Subtraktion mit einer Probe, die nur Rauschen enthält."""
    n_fft, hop = 512, 128                                   # 512 Samples = 32 ms bei 16000 Hz
    # 1) Kurzzeit-Spektren des verrauschten Signals: Betrag und Phase
    stft = librosa.stft(verrauscht, n_fft=n_fft, hop_length=hop)
    betrag, phase = np.abs(stft), np.angle(stft)
    # 2) mittleres Rauschspektrum aus der Rauschprobe
    rausch_betrag = np.abs(librosa.stft(rauschprobe, n_fft=n_fft, hop_length=hop)).mean(axis=1, keepdims=True)
    # 3) Rauschen abziehen, aber nie unter den Boden fallen
    neuer_betrag = np.maximum(betrag - staerke * rausch_betrag, boden * betrag)
    # 4) zurück in den Zeitbereich (die Phase des verrauschten Signals wird behalten)
    return librosa.istft(neuer_betrag * np.exp(1j * phase), hop_length=hop, length=len(verrauscht))

In [ ]:
staerke = 1.0     # ändern: 0.5, 1.0, 2.0, 4.0
boden = 0.05      # ändern: 0.01, 0.05, 0.2

entrauscht = spektrale_subtraktion(verrauscht, rauschprobe, staerke, boden)

print("SNR verrauscht:", round(snr_db(sprache, verrauscht), 1), "dB")
print("SNR entrauscht:", round(snr_db(sprache, entrauscht), 1), "dB")

skala = gemeinsame_skala(sprache, verrauscht, entrauscht)
anhoeren(sprache, sr_sprache, "Sauber:", skala)
anhoeren(verrauscht, sr_sprache, "Verrauscht:", skala)
anhoeren(entrauscht, sr_sprache, "Nach spektraler Subtraktion:", skala)

In [ ]:
von_s, bis_s = 3.0, 5.5          # gezeigter Zeitbereich in Sekunden

fig, achsen = plt.subplots(3, 1, figsize=(10, 6), sharex=True, sharey=True)   # 3 Diagramme untereinander
achsen[0].plot(zeit_sprache, sprache, linewidth=0.5)
achsen[0].set_title("Sauber")
achsen[1].plot(zeit_sprache, verrauscht, linewidth=0.5, color="tab:red")
achsen[1].set_title("Verrauscht")
achsen[2].plot(zeit_sprache, entrauscht, linewidth=0.5, color="tab:green")
achsen[2].set_title("Nach spektraler Subtraktion")
achsen[2].set_xlim(von_s, bis_s)
achsen[2].set_xlabel("Zeit [s]")
plt.tight_layout()
plt.show()

In [ ]:
f_s, db_s = spektrum(sprache, sr_sprache)
f_v, db_v = spektrum(verrauscht, sr_sprache)
f_e, db_e = spektrum(entrauscht, sr_sprache)

plt.figure(figsize=(10, 3))
plt.plot(f_v, db_v, label="verrauscht", color="tab:red", linewidth=0.8)
plt.plot(f_e, db_e, label="entrauscht", color="tab:green", linewidth=0.8)
plt.plot(f_s, db_s, label="sauber", color="tab:blue", linewidth=0.8)
plt.xlim(0, 8000)
plt.ylim(np.max(db_s) - 70, np.max(db_s) + 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Spektren im Vergleich")
plt.legend()
plt.grid(True)
plt.show()

> **Aufgabe 3.3:** Ändern Sie jeweils nur **einen** Parameter: `staerke` (0.5, 2.0, 4.0) oder `boden` (0.01, 0.2). Führen Sie danach auch `ziel_snr_db = 0` und `ziel_snr_db = 20` aus (dafür zuerst die Zelle in 3.3 und dann die Zellen in 3.4 erneut ausführen). Tragen Sie alles in **Tabelle D** ein.

> **Wichtig:** Ein höheres SNR bedeutet **nicht automatisch**, dass die Sprache angenehmer oder verständlicher klingt. Starke Rauschunterdrückung kann Sprache dumpf oder "blechern" machen und neue Artefakte erzeugen. Zu jeder Messung gehört deshalb auch ein **Hörvergleich**.

### 3.5 Lauter ist nicht besser

Wenn wir die verrauschte Aufnahme mit einem Faktor `verstaerkung` multiplizieren, werden **Sprache und Rauschen gleich stark** lauter. Das Verhältnis zwischen beiden bleibt gleich. Deshalb bleibt auch das SNR gleich.

Wird das Signal zu laut, werden alle Werte über 1 **abgeschnitten**. Das nennt man **Übersteuerung**. Sie erzeugt neue, hörbare Verzerrungen.

In [ ]:
verstaerkung = 4        # ändern: 2, 4, 10

lauter = verstaerkung * verrauscht
print("Spitzenwert nach Verstärkung:", round(np.max(np.abs(lauter)), 2))
print("SNR lauter (Sprache und Rauschen beide verstärkt):",
      round(snr_db(verstaerkung * sprache, lauter), 1), "dB")

uebersteuert = np.clip(lauter, -1, 1)       # Werte über 1 abschneiden, wie ein echter Wandler
print("Übersteuerte Samples:", np.sum(np.abs(lauter) > 1))
print("SNR nach Übersteuerung:", round(snr_db(verstaerkung * sprache, uebersteuert), 1), "dB")

In [ ]:
# Fairer Hörvergleich bei gleicher Lautstärke:
# Die übersteuerte Version wird dafür wieder durch die Verstärkung geteilt.
anhoeren(verrauscht, sr_sprache, "Verrauscht (ursprünglicher Pegel):")
anhoeren(uebersteuert / verstaerkung, sr_sprache, "Verstärkt und übersteuert, danach wieder leiser gemacht:")

> **Aufgabe 3.4:** Probieren Sie `verstaerkung = 2`, `4` und `10`. Ab welcher Verstärkung gibt es übersteuerte Samples? Wie klingt die Übersteuerung? Warum wird die Aufnahme durch Verstärken nicht "sauberer"?

**Ihre Antwort:** 

### 3.6 Musik mit einem Equalizer verändern

Ein **Equalizer** (wie am Mischpult) verstärkt oder dämpft einzelne Frequenzbereiche. Unser einfacher 3-Band-Equalizer teilt das Signal auf:
1. **Tiefen**: alles unter 250 Hz (Tiefpass)
2. **Höhen**: alles über 4000 Hz (Hochpass)
3. **Mitten**: der Rest (Original minus Tiefen minus Höhen)

Jedes Band wird mit einer Verstärkung in dB multipliziert. Dann werden die drei Bänder wieder addiert. +6 dB ist ungefähr die **doppelte** Amplitude, -6 dB ungefähr die **halbe**. Bei 0 dB bleibt das Band unverändert.

Der Equalizer ist eine **Klangveränderung**. Er ist kein Verfahren zur Rauschunterdrückung.

In [ ]:
def equalizer(x, sr, tiefen_db=0, mitten_db=0, hoehen_db=0):
    """3-Band-Equalizer: Signal aufteilen, jedes Band verstärken, wieder addieren."""
    tiefen = tiefpass(x, sr, 250)
    hoehen = hochpass(x, sr, 4000)
    mitten = x - tiefen - hoehen                 # der Rest ist das mittlere Band
    # dB in einen Faktor umrechnen: Faktor = 10 hoch (dB / 20)
    return (10**(tiefen_db / 20) * tiefen
            + 10**(mitten_db / 20) * mitten
            + 10**(hoehen_db / 20) * hoehen)

In [ ]:
tiefen_db = 6      # Tiefen in dB, zum Beispiel -12, 0, 6
mitten_db = 0      # Mitten in dB
hoehen_db = 0      # Höhen in dB

musik_eq = equalizer(musik_kurz, sr_musik, tiefen_db, mitten_db, hoehen_db)

skala = gemeinsame_skala(musik_kurz, musik_eq)
print("Gemeinsame Skala:", round(skala, 3))
anhoeren(musik_kurz, sr_musik, "Original:", skala)
anhoeren(musik_eq, sr_musik, "Mit Equalizer:", skala)

In [ ]:
# Frequenzgang des Equalizers (wieder mit einem Impuls gemessen)
antwort_eq = equalizer(impuls, sr_musik, tiefen_db, mitten_db, hoehen_db)
f_gang, db_gang = spektrum(antwort_eq, sr_musik, glaettung_hz=0)
db_gang = db_gang - 20 * np.log10(2 / len(antwort_eq))      # 0 dB = unverändert

plt.figure(figsize=(10, 3))
plt.semilogx(f_gang[1:], db_gang[1:])
plt.ylim(-15, 15)
plt.xlabel("Frequenz [Hz] (logarithmische Achse)")
plt.ylabel("Verstärkung [dB]")
plt.title("Frequenzgang des Equalizers")
plt.grid(True)
plt.show()

**Lauter oder besser?** Anhebungen machen die Musik meist auch **lauter**. Lauter klingt oft zuerst "besser", auch wenn der Klang nicht besser ist. Für einen fairen Vergleich bringen wir die Equalizer-Version auf **dieselbe mittlere Lautstärke (RMS)** wie das Original.

In [ ]:
rms_original = np.sqrt(np.mean(musik_kurz**2))
rms_eq = np.sqrt(np.mean(musik_eq**2))
musik_eq_gleich_laut = musik_eq * (rms_original / rms_eq)

print("RMS Original:", round(rms_original, 4), "  RMS Equalizer:", round(rms_eq, 4))
skala = gemeinsame_skala(musik_kurz, musik_eq_gleich_laut)
anhoeren(musik_kurz, sr_musik, "Original:", skala)
anhoeren(musik_eq_gleich_laut, sr_musik, "Equalizer, gleich laut wie das Original:", skala)

> **Aufgabe 3.5:** Stellen Sie die Kombinationen aus **Tabelle E** ein. Beschreiben Sie jeweils, wie die Musik klingt. Achten Sie beim Vergleich auf gleiche Lautstärke.

### 3.7 Optional: Ergebnis als WAV-Datei speichern

Die Datei wird im Ordner `ausgabe/` gespeichert. In Google Colab finden Sie sie links im Bereich **Dateien** (Ordnersymbol). Mit Rechtsklick können Sie sie herunterladen. Nach dem Zurücksetzen der Laufzeit ist sie gelöscht.

In [ ]:
os.makedirs("ausgabe", exist_ok=True)          # Ordner anlegen, falls noch nicht vorhanden

skala = gemeinsame_skala(entrauscht)           # verhindert Übersteuerung in der Datei
sf.write("ausgabe/sprache_entrauscht.wav", entrauscht * skala, sr_sprache, subtype="PCM_16")
print("Gespeichert: ausgabe/sprache_entrauscht.wav  (Skala:", round(skala, 3), ")")

## Auswertung TP3

**Tabelle C: Brummen und Hochpass**

| Grenzfrequenz [Hz] | SNR vorher [dB] | SNR nachher [dB] | Brummen noch hörbar? | Sprache verändert? |
|---|---|---|---|---|
| 60 | | | | |
| 120 | | | | |
| 200 | | | | |
| 400 | | | | |
| 800 | | | | |

**Tabelle D: Spektrale Subtraktion**

| ziel_snr_db | staerke | boden | SNR vorher [dB] | SNR nachher [dB] | Höreindruck |
|---|---|---|---|---|---|
| 10 | 1.0 | 0.05 | | | |
| 10 | 0.5 | 0.05 | | | |
| 10 | 2.0 | 0.05 | | | |
| 10 | 4.0 | 0.05 | | | |
| 10 | 1.0 | 0.01 | | | |
| 10 | 1.0 | 0.2 | | | |
| 0 | 1.0 | 0.05 | | | |
| 20 | 1.0 | 0.05 | | | |

**Tabelle E: Equalizer**

| Tiefen [dB] | Mitten [dB] | Höhen [dB] | Höreindruck |
|---|---|---|---|
| +6 | 0 | 0 | |
| -12 | 0 | 0 | |
| 0 | +6 | 0 | |
| 0 | -12 | 0 | |
| 0 | 0 | +6 | |
| 0 | 0 | -12 | |

**Fragen**
1. Warum ist ein Hochpass gegen Brummen geeignet, aber nicht gegen weißes Rauschen?
2. Warum sinkt das SNR wieder, wenn die Grenzfrequenz des Hochpasses zu hoch ist?
3. Bei welcher Einstellung war das SNR nach der spektralen Subtraktion am höchsten? Klang diese Version auch am besten? Begründen Sie.
4. Beschreiben Sie das "musikalische Rauschen". Bei welcher Einstellung war es am deutlichsten?
5. Warum verbessert eine Verstärkung das SNR nicht? Was passiert bei Übersteuerung?
6. Warum sollte man Equalizer-Einstellungen bei gleicher Lautstärke vergleichen?

**Ihre Antworten:**
1. 
2. 
3. 
4. 
5. 
6. 

# TP4: Sprache und Musik im Spektrogramm

## Lernziele

Nach diesem Versuch können Sie ...
- den Unterschied zwischen dem **Spektrum der ganzen Datei** (FFT) und einem **Spektrogramm** erklären,
- Spektrogramme von Sprache, Musik und Geräusch lesen und vergleichen,
- ein **Mel-Spektrogramm** einfach erklären,
- mit der **Kurzzeit-Energie** erkennen, **wann** gesprochen wird,
- die **Grenzen** dieser einfachen Methode benennen,
- Sprachaktivitätserkennung, Sprechererkennung und Spracherkennung unterscheiden.

## Vorbereitungsaufgaben

1. Ein Spektrogramm benutzt Fenster von 32 ms. Wie viele Samples sind das bei 16000 Hz?
2. Das Spektrum der ganzen Datei (TP2) zeigt nicht, **wann** eine Frequenz vorkommt. Warum ist das bei Sprache ein Nachteil?
3. Menschen hören den Unterschied zwischen 100 Hz und 200 Hz sehr deutlich, zwischen 8000 Hz und 8100 Hz aber kaum. Was bedeutet das für eine "gehörgerechte" Frequenzachse?
4. Erklären Sie mit eigenen Worten den Unterschied zwischen diesen drei Fragen: **Wann** spricht jemand? **Wer** spricht? **Was** wird gesagt?

**Ihre Antworten:**
1.
2.
3.
4.

## Durchführung

### 4.1 Von der FFT zum Spektrogramm

Die FFT der ganzen Datei (TP2) mischt alle Zeitpunkte zusammen. Ein **Spektrogramm** geht so vor:
1. Das Signal wird in kurze, überlappende **Fenster** zerlegt (zum Beispiel 32 ms).
2. Für jedes Fenster wird eine FFT berechnet.
3. Die Spektren werden nebeneinander als Bild gezeigt.

So lesen Sie das Bild: **x-Achse = Zeit**, **y-Achse = Frequenz**, **Farbe = Stärke** (hell = stark, dunkel = schwach).

`n_fft` ist die Fensterlänge in Samples. `hop` ist der Schritt von einem Fenster zum nächsten.

In [ ]:
import librosa.display

def spektrogramm_zeigen(x, sr, titel, n_fft=1024, hop=256, ax=None):
    """Zeigt ein Spektrogramm. Stereo wird nur für die Darstellung zu Mono gemittelt."""
    if x.ndim == 2:
        x = x.mean(axis=1)
    stft = librosa.stft(x, n_fft=n_fft, hop_length=hop)             # Kurzzeit-FFT
    stft_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)     # in dB, 0 dB = stärkster Punkt
    neue_figur = ax is None
    if neue_figur:
        plt.figure(figsize=(10, 3))
        ax = plt.gca()
    bild = librosa.display.specshow(stft_db, sr=sr, hop_length=hop, x_axis="time", y_axis="hz",
                                    cmap="magma", vmin=-80, vmax=0, ax=ax)
    plt.colorbar(bild, ax=ax, format="%+2.0f dB")
    ax.set_xlabel("Zeit [s]")
    ax.set_ylabel("Frequenz [Hz]")
    ax.set_title(titel)
    if neue_figur:
        plt.show()

In [ ]:
n_fft = 512      # Fensterlänge in Samples (512 Samples bei 16000 Hz = 32 ms)
hop = 128        # Schritt in Samples

f1, db1 = spektrum(sprache, sr_sprache)
plt.figure(figsize=(10, 3))
plt.plot(f1, db1 - np.max(db1), linewidth=0.8)
plt.xlim(0, 8000)
plt.ylim(-80, 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB]")
plt.title("Ganze Datei: FFT-Spektrum (keine Zeitinformation)")
plt.grid(True)
plt.show()

spektrogramm_zeigen(sprache, sr_sprache, "Spektrogramm: Frequenzen über der Zeit", n_fft, hop)

> **Aufgabe 4.1:**
> a) Finden Sie im Spektrogramm die **Pausen** (dunkle senkrechte Streifen). Passen sie zu Aufgabe 0.5?
> b) In Vokalen sieht man viele **waagerechte Linien** übereinander. Was sind diese Linien? (Tipp: Aufgabe 2.8)
> c) Wo sehen Sie Laute wie "s" oder "sch" (Energie vor allem bei hohen Frequenzen)?
> d) Setzen Sie `n_fft = 128` und danach `n_fft = 2048`. Was wird schärfer: die Zeit (senkrecht) oder die Frequenz (waagerecht)?

**Ihre Antwort:** 

### 4.2 Sprache, Musik und Geräusch im Vergleich

In [ ]:
fig, achsen = plt.subplots(3, 1, figsize=(10, 9))
spektrogramm_zeigen(sprache, sr_sprache, "Sprache", ax=achsen[0])
spektrogramm_zeigen(musik, sr_musik, "Musik (für die Darstellung zu Mono gemittelt)", ax=achsen[1])
spektrogramm_zeigen(geraeusch, sr_geraeusch, "Rennwagen (für die Darstellung zu Mono gemittelt)", ax=achsen[2])
plt.tight_layout()
plt.show()

> **Aufgabe 4.2:** Beschreiben Sie in **Tabelle G** das typische Muster jeder Aufnahme. Hinweise: Gibt es Pausen? Lange waagerechte Linien (gehaltene Töne)? Senkrechte Striche (Schläge, Knacken)? Linien, die steigen oder fallen (Tonhöhe ändert sich)? Breites Rauschen über viele Frequenzen?

### 4.3 Das Mel-Spektrogramm

Das menschliche Gehör unterscheidet **tiefe** Frequenzen sehr fein und **hohe** Frequenzen eher grob. Die **Mel-Skala** bildet das nach: Unten sind die Frequenzbänder schmal, oben breit.

Ein **Mel-Spektrogramm** fasst die vielen FFT-Frequenzen zu wenigen **Mel-Bändern** zusammen (zum Beispiel 64). Das Bild wird kleiner, behält aber das, was für das Hören wichtig ist. Mel-Spektrogramme sind eine typische **Eingabe für moderne KI-Modelle**, zum Beispiel für neuronale Netze (CNNs), die Audio wie ein Bild verarbeiten.

In [ ]:
n_mels = 64      # Anzahl der Mel-Bänder, zum Beispiel 20, 64 oder 128

mel = librosa.feature.melspectrogram(y=sprache, sr=sr_sprache, n_fft=512, hop_length=128, n_mels=n_mels)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(10, 3))
bild = librosa.display.specshow(mel_db, sr=sr_sprache, hop_length=128, x_axis="time", y_axis="mel",
                                cmap="magma", vmin=-80, vmax=0)
plt.colorbar(bild, format="%+2.0f dB")
plt.xlabel("Zeit [s]")
plt.ylabel("Frequenz [Hz] (Mel-Skala)")
plt.title("Mel-Spektrogramm der Sprache")
plt.show()

print("Normales Spektrogramm:", 512 // 2 + 1, "Frequenzen pro Zeitfenster")    # // bedeutet ganzzahlig teilen
print("Mel-Spektrogramm:     ", mel_db.shape[0], "Mel-Bänder pro Zeitfenster")

> **Aufgabe 4.3:** Setzen Sie `n_mels = 20` und danach `n_mels = 128`. Wie verändert sich das Bild? Sind die Pausen und die Vokale noch zu erkennen?

**Ihre Antwort:** 

### 4.4 Sprachaktivität erkennen mit der Kurzzeit-Energie

Wir wollen herausfinden, **wann** in `speech.wav` gesprochen wird. Diese Aufgabe heißt **Sprachaktivitätserkennung** (englisch *Voice Activity Detection*, VAD).

So funktioniert die einfache Methode:
1. Das Signal wird in kurze Fenster zerlegt (25 ms lang, Schritt 10 ms).
2. Für jedes Fenster wird die **Energie** berechnet: der Mittelwert der quadrierten Samples.
3. Die Energie wird in dB umgerechnet. 0 dB ist das lauteste Fenster der Aufnahme.
4. Fenster **über** der Schwelle `schwelle_db` gelten als **aktiv** (Sprache). Fenster darunter gelten als **inaktiv** (Pause).

In [ ]:
def kurzzeit_energie_db(x, sr, fenster_ms=25, schritt_ms=10):
    """Energie pro kurzem Fenster in dB (0 dB = lautestes Fenster)."""
    fenster = sekunden_zu_samples(fenster_ms / 1000, sr)
    schritt = sekunden_zu_samples(schritt_ms / 1000, sr)
    energie = []
    zeiten = []
    for start in range(0, len(x) - fenster, schritt):      # start = 0, schritt, 2 mal schritt, ...
        stueck = x[start:start + fenster]
        energie.append(np.mean(stueck**2))
        zeiten.append((start + fenster / 2) / sr)          # Mitte des Fensters in Sekunden
    energie = np.array(energie)
    energie_db = 10 * np.log10(energie / np.max(energie) + 1e-12)
    return np.array(zeiten), energie_db

In [ ]:
def aktive_intervalle(zeiten, aktiv):
    """Fasst aufeinanderfolgende aktive Fenster zu Abschnitten (Beginn, Ende) zusammen."""
    intervalle = []
    beginn = None                         # None heißt: gerade kein aktiver Abschnitt
    for i in range(len(aktiv)):
        if aktiv[i] and beginn is None:   # ein aktiver Abschnitt beginnt
            beginn = zeiten[i]
        if not aktiv[i] and beginn is not None:   # ein aktiver Abschnitt endet
            intervalle.append((beginn, zeiten[i]))
            beginn = None
    if beginn is not None:                # letzter Abschnitt läuft bis zum Ende
        intervalle.append((beginn, zeiten[-1]))
    return intervalle

In [ ]:
def sprachaktivitaet(x, sr, schwelle_db, titel):
    """Erkennt aktive Abschnitte, zeichnet sie und gibt sie als Liste zurück."""
    zeiten, energie_db = kurzzeit_energie_db(x, sr)
    aktiv = energie_db > schwelle_db
    intervalle = aktive_intervalle(zeiten, aktiv)

    fig, achsen = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    achsen[0].plot(np.arange(len(x)) / sr, x, linewidth=0.5)
    for beginn_s, ende_s in intervalle:                    # jeden Abschnitt orange markieren
        achsen[0].axvspan(beginn_s, ende_s, color="orange", alpha=0.3)
    achsen[0].set_title(titel + " (orange = als aktiv erkannt)")
    achsen[1].plot(zeiten, energie_db)
    achsen[1].axhline(schwelle_db, color="red", linestyle="--", label="Schwelle")
    achsen[1].set_ylim(-80, 5)
    achsen[1].set_xlabel("Zeit [s]")
    achsen[1].set_ylabel("Energie [dB]")
    achsen[1].legend(loc="lower right")
    plt.show()

    print("Anzahl aktiver Abschnitte:", len(intervalle))
    print("Anteil aktiv:", round(100 * np.mean(aktiv), 1), "%")
    return intervalle

In [ ]:
schwelle_db = -30     # ändern: -10, -20, -30, -40, -50

intervalle = sprachaktivitaet(sprache, sr_sprache, schwelle_db, "Sprache")

In [ ]:
# Die ersten 10 erkannten Abschnitte (Beginn und Ende in Sekunden)
for beginn_s, ende_s in intervalle[:10]:
    print("aktiv von", round(beginn_s, 2), "s bis", round(ende_s, 2), "s")

In [ ]:
# Nur die aktiven Abschnitte hintereinander anhören (die erkannten Pausen fallen weg)
teile = []
for beginn_s, ende_s in intervalle:
    teile.append(sprache[sekunden_zu_samples(beginn_s, sr_sprache):sekunden_zu_samples(ende_s, sr_sprache)])

if len(teile) > 0:
    nur_aktiv = np.concatenate(teile)          # alle Teile aneinanderhängen
    print("Original:", round(len(sprache) / sr_sprache, 1), "s   nur aktiv:", round(len(nur_aktiv) / sr_sprache, 1), "s")
    anhoeren(nur_aktiv, sr_sprache, "Nur die als aktiv erkannten Abschnitte:")
else:
    print("Kein aktiver Abschnitt gefunden. Bitte eine niedrigere Schwelle wählen.")

> **Aufgabe 4.4:** Probieren Sie `schwelle_db = -10`, `-20`, `-30`, `-40` und `-50`. Führen Sie jeweils die drei Zellen oben aus. Tragen Sie die Ergebnisse in **Tabelle F** ein. Bei welcher Schwelle werden Wortanfänge oder Wortenden abgeschnitten? Bei welcher Schwelle zählen auch Pausen als aktiv?

### 4.5 Grenzen der Methode

**a) Rauschen kann die Erkennung auslösen.** Wir benutzen die verrauschte Sprache aus TP3 mit derselben Schwelle.

In [ ]:
intervalle_rauschen = sprachaktivitaet(verrauscht, sr_sprache, schwelle_db, "Verrauschte Sprache aus TP3")

**b) Leise Sprache kann übersehen werden.** Wir machen die zweite Hälfte der Aufnahme etwa 30 dB leiser. Die Schwelle bezieht sich auf das lauteste Fenster der ganzen Aufnahme.

In [ ]:
leiser_teil = sprache.copy()
mitte = len(sprache) // 2
leiser_teil[mitte:] = leiser_teil[mitte:] * 0.03     # Faktor 0.03 = etwa -30 dB

intervalle_leise = sprachaktivitaet(leiser_teil, sr_sprache, schwelle_db, "Zweite Hälfte sehr leise")
anhoeren(leiser_teil, sr_sprache, "Zweite Hälfte sehr leise:")

> **Zusammengefasst:** Eine Energie-Schwelle unterscheidet nur zwischen "laut" und "leise". Laute Geräusche wie Rauschen, Husten, Türen oder Musik können als Sprache gelten. Leise Sprache kann übersehen werden. Bessere Verfahren benutzen zusätzlich das Spektrum oder trainierte Modelle.

> **Aufgabe 4.5:** Suchen Sie eine Schwelle, die bei der **verrauschten** Sprache die Pausen wieder erkennt. Funktioniert dieselbe Schwelle auch für die Aufnahme mit der **leisen** zweiten Hälfte? Was folgt daraus?

**Ihre Antwort:** 

### 4.6 Wann, wer, was? Drei verschiedene Aufgaben

| Aufgabe | Frage | Beispiel für ein Ergebnis | In diesem Labor? |
|---|---|---|---|
| **Sprachaktivitätserkennung** (VAD) | **Wann** wird gesprochen? | "aktiv von 0,4 s bis 2,0 s" | Ja, mit der Kurzzeit-Energie |
| **Sprechererkennung** (englisch *speaker identification*) | **Wer** spricht? | "Person A" | Nein |
| **Spracherkennung** (englisch *automatic speech recognition*, ASR) | **Was** wird gesagt? | der Text des Satzes | Nein |

- Die Energie-Schwelle sagt **nichts** darüber, **wer** spricht oder **was** gesagt wird. Sie misst nur, ob ein Abschnitt laut genug ist.
- `speech.wav` enthält **eine einzige** Sprecherstimme. Mit dieser Aufnahme kann man Sprechererkennung weder üben noch testen. Dafür bräuchte man Aufnahmen **mehrerer** bekannter Personen.
- Sprechererkennung und Spracherkennung arbeiten mit trainierten Modellen. Als Eingabe dienen oft Mel-Spektrogramme oder MFCCs (siehe 4.8).

### 4.7 Optional: Echo auf der Musik

Ein **Echo** ist eine verzögerte, leisere Kopie des Signals:

Ausgang = Original + `daempfung` mal (Original, um `verzoegerung_ms` später)

Beide Stereokanäle bekommen dasselbe Echo. Das Stereobild bleibt also erhalten.

In [ ]:
verzoegerung_ms = 300    # Verzögerung in Millisekunden, zum Beispiel 50, 300, 1000
daempfung = 0.5          # Lautstärke des Echos, zum Beispiel 0.3 oder 0.8

verzoegerung = sekunden_zu_samples(verzoegerung_ms / 1000, sr_musik)
mit_echo = np.zeros((len(musik_kurz) + verzoegerung, 2))    # Platz für das Echo am Ende
mit_echo[:len(musik_kurz)] += musik_kurz                    # Original
mit_echo[verzoegerung:] += daempfung * musik_kurz           # verzögerte, leisere Kopie

skala = gemeinsame_skala(musik_kurz, mit_echo)
print("Gemeinsame Skala:", round(skala, 3))
anhoeren(musik_kurz, sr_musik, "Original:", skala)
anhoeren(mit_echo, sr_musik, "Mit Echo:", skala)

> **Aufgabe 4.6 (optional):** Vergleichen Sie `verzoegerung_ms = 50`, `300` und `1000` sowie `daempfung = 0.3` und `0.8`. Wann hören Sie ein deutliches Echo? Wann klingt es eher wie ein Raum oder eine Klangfärbung?

**Ihre Antwort:** 

### 4.8 Optional: MFCC, kompakte Merkmale für KI-Modelle

**MFCCs** (Mel-Frequency Cepstral Coefficients) fassen jedes Fenster des Mel-Spektrogramms in wenigen Zahlen zusammen, hier 13. Sie beschreiben grob die **Klangfarbe**.

- MFCCs sind klassische Eingaben für **Maschinelles Lernen**, zum Beispiel für die Erkennung gesprochener Wörter.
- Mel-Spektrogramme sind typische Eingaben für **CNNs**, die das "Bild" des Klangs verarbeiten.

Wir berechnen die MFCCs hier nur. Es wird **kein** Modell trainiert.

In [ ]:
mfcc = librosa.feature.mfcc(y=sprache, sr=sr_sprache, n_mfcc=13, n_fft=512, hop_length=160)

plt.figure(figsize=(10, 3))
bild = librosa.display.specshow(mfcc, sr=sr_sprache, hop_length=160, x_axis="time", cmap="viridis")
plt.colorbar(bild)
plt.xlabel("Zeit [s]")
plt.ylabel("MFCC-Nummer")
plt.title("13 MFCCs der Sprache")
plt.show()

print("Form:", mfcc.shape, "= (13 Koeffizienten, Anzahl Zeitfenster)")

## Auswertung TP4

**Tabelle F: Schwelle der Sprachaktivitätserkennung (`speech.wav`)**

| schwelle_db | Anzahl aktiver Abschnitte | Anteil aktiv [%] | Beobachtung (Wörter abgeschnitten? Pausen als aktiv?) |
|---|---|---|---|
| -10 | | | |
| -20 | | | |
| -30 | | | |
| -40 | | | |
| -50 | | | |

**Tabelle G: Muster im Spektrogramm**

| Aufnahme | Typisches Muster |
|---|---|
| Sprache | |
| Musik | |
| Rennwagen | |

**Fragen**
1. Was zeigt ein Spektrogramm, was das FFT-Spektrum der ganzen Datei nicht zeigt?
2. Was ändert sich, wenn `n_fft` größer wird? Erklären Sie mit Zeit und Frequenz.
3. Warum passt die Mel-Skala besser zum menschlichen Hören als eine lineare Frequenzachse?
4. Welche Schwelle war für `speech.wav` am besten? Warum funktioniert sie bei der verrauschten Sprache nicht?
5. Warum kann eine Energie-Schwelle **nicht** erkennen, **wer** spricht?
6. Nennen Sie eine Anwendung, in der Sprachaktivitätserkennung nützlich ist.

**Ihre Antworten:**
1. 
2. 
3. 
4. 
5. 
6. 

---
## Quellen der Audiodaten

Die drei Aufnahmen stammen von Dritten. Für dieses Labor wurden nur Ausschnitte herausgeschnitten und in WAV-Dateien umgewandelt (mit 20 ms Ein- und Ausblenden an den Schnittstellen, sonst ohne Bearbeitung).

- **`speech.wav`**: LibriVox-Hörbuch "Fabeln" von Magnus Gottfried Lichtwer, Abschnitt "Das Wiesel und die Hühner", gelesen von **marham63**. Lizenz: **Public Domain**. Quelle: <https://archive.org/details/fabeln_1905_librivox>. Ausschnitt 12,8 s bis 28,7 s, umgetastet auf 16000 Hz, mono.
- **`music.wav`**: Julius Fučík, "Entrance of the Gladiators" ("Einzug der Gladiatoren"), gespielt von der **United States Marine Band**. Lizenz: **Public Domain** (Werk der US-Regierung). Quelle: <https://commons.wikimedia.org/wiki/File:Julius_Fu%C4%8D%C3%ADk%27s_%22Entrance_of_the_Gladiators%22,_performed_by_the_United_States_Marine_Band.flac>. Ausschnitt 24 s bis 44 s, 44100 Hz, stereo.
- **`sound.wav`**: "Audi R8 (2000)", Rennwagen beim Goodwood Festival of Speed 2009, aufgenommen von **Edvvc**. Lizenz: **CC BY-SA 3.0** (<https://creativecommons.org/licenses/by-sa/3.0/>). Quelle: <https://commons.wikimedia.org/wiki/File:Audi_R8_(2000).ogg>. Ausschnitt 4 s bis 12 s, umgetastet auf 22050 Hz, stereo. Der gekürzte Ausschnitt steht ebenfalls unter CC BY-SA 3.0.